# Lecture 14 Activity – Does the 1947 House vote on one dimension or two?

QSS 20 · Modern Statistical Computing · Fall 2026

## The question

In the lecture, PCA on the roll calls of every House since 2013 found
one direction that splits the parties and explains 72 to 83 percent of
the variance. The 80th House (1947 to 1948) was different. Republicans
controlled both chambers of Congress for the first time since 1931, and
the Democratic Party held both the segregated South and the northern
cities.

> **From the votes alone, how many dimensions does the 80th House need,
> and does a second dimension separate the southern Democrats?**

### How to work

Work in pairs for 30 minutes. Submit the matrix shape, the scree plot, the
two correlations, the cluster crosstab, the table of roll calls, and a
short claim. If you run out of time, finish Task 6 after class.

- Each row of the matrix is **one member of the 80th House**, and each
  column is a roll call. The setup reads the three Voteview files, keeps
  the House members of the two parties, and marks the members from the
  eleven states of the former Confederacy in `south`.
- Use the lecture's settings: drop members with fewer than half the
  votes, fill missing votes with the roll call's share of yeas, and use
  `n_init=20` and `random_state=42`.
- In the starter cells, replace each `...` with code.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

pd.set_option('display.max_colwidth', 70)

VOTEVIEW = Path('../../data/raw/voteview')
votes = pd.read_csv(VOTEVIEW / 'H080_votes.csv.gz')
members = pd.read_csv(VOTEVIEW / 'H080_members.csv')
rollcalls = pd.read_csv(VOTEVIEW / 'H080_rollcalls.csv')

# House members of the two parties. This leaves out the president and
# one member of the American Labor Party.
members = members[(members['chamber'] == 'House')
                  & members['party_code'].isin([100, 200])].copy()
members['party'] = members['party_code'].map({100: 'Democrat',
                                               200: 'Republican'})
south = ['AL', 'AR', 'FL', 'GA', 'LA', 'MS', 'NC', 'SC', 'TN', 'TX',
         'VA']
members['south'] = members['state_abbrev'].isin(south)
members['group'] = members['party'].replace('Democrat',
                                            'Northern Democrat')
southern = (members['party'] == 'Democrat') & members['south']
members.loc[southern, 'group'] = 'Southern Democrat'
votes = votes[votes['icpsr'].isin(members['icpsr'])]

# Check: 69,950 votes; 447 members by group.
print(votes.shape)
members['group'].value_counts()

## Task 1 · Build the vote matrix

Code each vote as in the lecture: yea and stated yea become 1, nay and
stated nay become 0, and everything else is missing. Pivot to one row per
member and one column per roll call, then print the shape and the share
of missing cells. Keep the members who voted on at least half of the
roll calls, then fill each missing vote with the roll call's share of
yeas.

How many roll calls did the 80th House take, and how many members remain?



<details><summary>Hint</summary>

`votes['cast_code'].map(yea_or_nay)`, then
`votes.pivot(index='icpsr', columns='rollnumber', values='yea')`. The
share of votes a member cast is `matrix.notna().mean(axis=1)`; the fill
is `matrix.fillna(matrix.mean())`.

</details>

In [ ]:
yea_or_nay = {1: 1, 2: 1, 3: 1, 4: ..., 5: ..., 6: ...}
votes['yea'] = votes['cast_code'].map(...)
matrix = votes.pivot(index=..., columns=..., values=...)
print(matrix.shape, round(matrix.isna().mean().mean(), 3))
voted = matrix.notna().mean(axis=1)
matrix = matrix[voted >= ...]
filled = matrix.fillna(...)
filled.shape

*Your answer:*

## Task 2 · How much of the variance does each component explain?

Fit `PCA` with ten components on `filled`, draw the shares as a bar
chart, and print the first five.

In the lecture, PC1 explained 0.72 to 0.83 of the variance in the
Houses since 2013, and PC2 explained 0.025 to 0.051. Where does 1947
fall, and after which component does the scree plot bend?



<details><summary>Hint</summary>

`pca = PCA(n_components=10, random_state=0)`,
`scores = pca.fit_transform(filled)`, then
`ax.bar(range(1, 11), pca.explained_variance_ratio_)`.

</details>

In [ ]:
pca = PCA(n_components=..., random_state=0)
scores = pca.fit_transform(...)
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(range(1, 11), ...)
ax.set(xlabel='Component', ylabel='Share of the variance',
       xticks=range(1, 11))
plt.show()
pca.explained_variance_ratio_[:5].round(3)

*Your answer:*

## Task 3 · Compare the components with NOMINATE

Put the first two PCA scores next to each member's group and NOMINATE
scores. Print the correlation of PC1 with `nominate_dim1` and of PC2
with `nominate_dim2`, and the mean of each score in each group.

Which component separates the southern Democrats? On PC1, are they
closer to the northern Democrats or to the Republicans?



<details><summary>Hint</summary>

`house = members.set_index('icpsr').loc[filled.index].copy()` lines the
members up with the rows of `filled`. A negative correlation means the
component points the opposite way from NOMINATE: the sign of a component
is arbitrary.

</details>

In [ ]:
house = members.set_index('icpsr').loc[filled.index].copy()
house['pc1'] = scores[:, ...]
house['pc2'] = scores[:, ...]
print(round(house['pc1'].corr(house[...]), 3))
print(round(house['pc2'].corr(house[...]), 3))
house.groupby('group')[['pc1', 'pc2', 'nominate_dim1',
                        'nominate_dim2']].mean().round(2)

In [ ]:
group_colors = {'Northern Democrat': 'C0', 'Southern Democrat': 'C1',
                'Republican': 'C3'}
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for group, color in group_colors.items():
    part = house[house['group'] == group]
    ax.scatter(part[...], part[...], s=12, color=color, alpha=0.7,
               label=group)
ax.set(xlabel='Score on PC1', ylabel='Score on PC2')
ax.legend(fontsize=8)
plt.show()

*Your answer:*

## Task 4 · Cluster the members on the two components

Print the inertia and the silhouette for $k$ from 2 to 6 on the two
scores `house[['pc1', 'pc2']]`. Then fit three clusters, store them in
`house['cluster']`, and crosstab them against `group`. Last, list the
states of the northern Democrats who share a cluster with the southern
Democrats.

Which $k$ does the silhouette favor? Describe the three clusters in
terms of party and region.



<details><summary>Hint</summary>

Copy the lecture's loop over $k$. For the last step, find the cluster
that holds most southern Democrats with
`house.loc[southern_rows, 'cluster'].mode().iloc[0]`, then filter
`group == 'Northern Democrat'` and that cluster, and call
`['state_abbrev'].value_counts()`.

</details>

In [ ]:
two_scores = house[['pc1', 'pc2']]
for k in range(2, 7):
    km_k = KMeans(n_clusters=k, n_init=20, random_state=42)
    km_k.fit(two_scores)
    print(f'k = {k}   inertia {km_k.inertia_:6.1f}   silhouette '
          f'{silhouette_score(two_scores, km_k.labels_):.3f}')

In [ ]:
km3 = KMeans(n_clusters=..., n_init=20, random_state=42)
house['cluster'] = km3.fit_predict(...)
pd.crosstab(house['group'], house['cluster'])

In [ ]:
southern_rows = house['group'] == 'Southern Democrat'
south_cluster = house.loc[southern_rows, 'cluster'].mode().iloc[0]
with_south = house[(house['group'] == ...)
                   & (house['cluster'] == south_cluster)]
with_south['state_abbrev'].value_counts()

*Your answer:*

## Task 5 · Which roll calls does PC2 follow?

List the three roll calls with the largest positive weights on PC2 and
the three with the largest negative weights. Show each one's date,
description, and weight, and the share of each group that voted yea.

What do the roll calls at the two ends of PC2 have in common?



<details><summary>Hint</summary>

Put the PC2 weights in a Series indexed by roll call:
`pd.Series(pca.components_[1], index=filled.columns)`. As in the
lecture, `filled.groupby(...).mean().T` gives the yea shares, and
`.nlargest(3)` and `.nsmallest(3)` pick the roll calls.
`rollcalls.set_index('rollnumber')` joins the descriptions.

</details>

In [ ]:
weights = pd.Series(pca.components_[...], index=filled.columns,
                    name='weight').round(3)
yea_share = filled.groupby(house['group']).mean().T.round(2)
about = rollcalls.set_index('rollnumber')[['date', 'dtl_desc']]
about['dtl_desc'] = about['dtl_desc'].str[:60]
top = weights.nlargest(...).index.append(weights.nsmallest(...).index)
about.loc[top].join(weights).join(yea_share)

*Your answer:*

## Task 6 · Make a claim

Write four or five sentences. Say how many dimensions the 1947 votes
need and which number shows it. Say in plain words what PC1 and PC2
measure, and how the 1947 House differs from the Houses since 2013. Name
the three clusters and the score that supports $k = 3$. Then state one
thing that neither PCA nor clustering can establish about these members.

*Your answer:*